In [0]:
%sql
-- Gold 3_12 — lead_emails: the CLIENT email view, built on the 2_8 identity spine.
-- v3 (Nov 2 2026): staff separation + primary email.
--   * customer_emails = emails of the lead's attributed persons MINUS every email owned
--     by a staff person (gold.staff_emails — whole-master rule: a 48-char organizer login,
--     an @dataengineeracademy.com address, a close_crm registration, or Calendly event
--     hosting makes the entire person staff, masked emails included).
--   * primary_customer_email = THE one client email, chosen by priority:
--       1. a REAL email from the lead's own CRM contact record
--       2. any REAL non-staff email of the lead's attributed persons
--       3. a MASKED email from the lead's own CRM contact record
--       4. any MASKED non-staff email
--     (alphabetical tie-break inside each priority; NULL only if every email
--     the lead ever touched was staff).
--   * non_primary_customer_emails = the remaining searchable set (masked correspondents,
--     alternate personal addresses).
-- Search: email = primary_customer_email OR array_contains(customer_emails, email).
-- The spine (silver.user_identity_map) remains the everyone-lookup, staff included;
-- this table is the client view. gold.staff_emails is the staff view.
CREATE OR REPLACE TABLE crm_ingestion.gold.lead_emails AS (
    WITH staff AS (
        SELECT DISTINCT e AS email
        FROM crm_ingestion.gold.staff_emails se
        LATERAL VIEW explode(se.all_emails) e AS e
    ),
    -- the lead's own CRM contact record: the authoritative source for the primary email
    lead_own AS (
        SELECT DISTINCT c.leads_processed_id AS lead_id, LOWER(TRIM(ce.email)) AS email,
               (LOWER(TRIM(ce.email)) NOT LIKE '%@masked.com') AS is_real
        FROM crm_ingestion.silver.leads_processed_contacts c
        JOIN crm_ingestion.silver.leads_processed_contacts_emails ce
          ON ce.leads_processed_contacts_id = c.id
        WHERE ce.email IS NOT NULL AND ce.email != ''
    ),
    person_emails AS (
        SELECT explode(sp.lead_ids) AS lead_id, e.email AS email,
               (e.email NOT LIKE '%@masked.com') AS is_real
        FROM crm_ingestion.silver.user_identity_map sp
        LATERAL VIEW explode(concat(sp.real_emails, sp.masked_emails)) e AS email
        WHERE e.email IS NOT NULL AND e.email != ''
    ),
    pe_marked AS (
        SELECT pe.lead_id, pe.email, pe.is_real,
               (s.email IS NOT NULL) AS is_staff,
               (o.lead_id IS NOT NULL) AS is_own
        FROM person_emails pe
        LEFT JOIN staff s ON s.email = pe.email
        LEFT JOIN (SELECT DISTINCT lead_id, email FROM lead_own) o
               ON o.lead_id = pe.lead_id AND o.email = pe.email
    ),
    base AS (
        SELECT lead_id,
               collect_set(CASE WHEN NOT is_staff THEN email END) AS customer_emails
        FROM pe_marked
        GROUP BY lead_id
    ),
    pick AS (
        SELECT lead_id, email FROM (
            SELECT lead_id, email,
                   ROW_NUMBER() OVER (
                       PARTITION BY lead_id
                       ORDER BY CASE WHEN is_real AND is_own THEN 1
                                     WHEN is_real          THEN 2
                                     WHEN is_own           THEN 3
                                     ELSE 4 END,
                                email
                   ) AS rn
            FROM pe_marked WHERE NOT is_staff
        ) WHERE rn = 1
    )
    SELECT
        b.lead_id,
        p.email AS primary_customer_email,
        CASE WHEN p.email IS NOT NULL
             THEN array_except(sort_array(b.customer_emails), ARRAY(p.email))
             ELSE sort_array(b.customer_emails)
        END AS non_primary_customer_emails,
        sort_array(b.customer_emails) AS customer_emails
    FROM base b
    LEFT JOIN pick p ON p.lead_id = b.lead_id
)

In [0]:
# Population funnel — every "how many leads/clients have emails" count in ONE place.
# Each row answers a DIFFERENT question; the counts differ by design:
#   lead_emails is the phone book for ALL leads; lead_dim / the profile are
#   the report card for CURRENT Coaching Clients only.
funnel = spark.sql("""
    SELECT '1. CRM leads total (silver, distinct lead IDs)' AS step, COUNT(DISTINCT id) AS count
    FROM crm_ingestion.silver.leads_processed
    UNION ALL
    SELECT '2. lead_emails rows (leads with any attributed email)', COUNT(*)
    FROM crm_ingestion.gold.lead_emails
    UNION ALL
    SELECT '3.    of which have a primary email', COUNT(primary_customer_email)
    FROM crm_ingestion.gold.lead_emails
    UNION ALL
    SELECT '4. Current Coaching Clients (lead_dim)', COUNT(*)
    FROM crm_ingestion.gold.lead_dim
    UNION ALL
    SELECT '5. Clients present in lead_emails (must equal row 4)', COUNT(*)
    FROM crm_ingestion.gold.lead_dim d
    JOIN crm_ingestion.gold.lead_emails e ON e.lead_id = d.lead_id
    UNION ALL
    SELECT '6. Clients with a primary email', COUNT(e.primary_customer_email)
    FROM crm_ingestion.gold.lead_dim d
    JOIN crm_ingestion.gold.lead_emails e ON e.lead_id = d.lead_id
    UNION ALL
    SELECT '7. Clients with NO primary (all emails staff-owned/absent)', COUNT(*)
    FROM crm_ingestion.gold.lead_dim d
    JOIN crm_ingestion.gold.lead_emails e ON e.lead_id = d.lead_id
    WHERE e.primary_customer_email IS NULL
    UNION ALL
    SELECT '8. Distinct primary emails among clients (shared-email effect)', COUNT(DISTINCT e.primary_customer_email)
    FROM crm_ingestion.gold.lead_dim d
    JOIN crm_ingestion.gold.lead_emails e ON e.lead_id = d.lead_id
    UNION ALL
    SELECT '9. Profile open versions (must equal row 4)', COUNT(*)
    FROM crm_ingestion.gold.customer_health_profile
    WHERE VRSN_END_DATE IS NULL
""")
display(funnel)

In [0]:
%sql
-- Validation: ZERO STAFF EMAILS IN THE CLIENT VIEW. The build subtracts
-- staff_emails.all_emails (the 189 spine staff persons). This proves the
-- guarantee is airtight BEYOND that list too: every one of the 443 close_crm
-- directory emails (staff_dim.csm_email, including the ~258 users with no
-- spine footprint) must also be absent — a directory email can only appear
-- on a lead via the spine, and a registered email in the spine fires the
-- is_registered staff marker, making the whole person staff and subtracting
-- every email they own. Any nonzero leak count here is a real leak.
WITH cust AS (
    SELECT DISTINCT e.email
    FROM crm_ingestion.gold.lead_emails le
    LATERAL VIEW explode(le.customer_emails) e AS email
),
blocklist AS (
    SELECT DISTINCT e AS email
    FROM crm_ingestion.gold.staff_emails se
    LATERAL VIEW explode(se.all_emails) e AS e
)
SELECT
    (SELECT COUNT(*) FROM cust)                                AS distinct_customer_emails,
    (SELECT COUNT(*) FROM cust c JOIN blocklist b ON b.email = c.email)
                                                               AS staff_blocklist_leaks,
    (SELECT COUNT(*) FROM cust c
      JOIN crm_ingestion.gold.staff_dim d ON d.csm_email = c.email)
                                                               AS directory_email_leaks